In [1]:
#imports
import pandas as pd
pd.set_option('display.max_columns', None)

In [2]:
crash_df = pd.read_csv('datasets/Motor_Vehicle_Collisions_-_Crashes_20260407.csv', delimiter=',')
vehicle_df = pd.read_csv('datasets/Motor_Vehicle_Collisions_-_Vehicles_20260503.csv', delimiter=',')
person_df = pd.read_csv('datasets/Motor_Vehicle_Collisions_-_Person_20260503.csv', delimiter=',')

/var/folders/tb/jb06w8hn5r72136sx1ztbxnr0000gn/T/ipykernel_35980/1986786415.py:1: DtypeWarning: Columns (0: ZIP CODE) have mixed types. Specify dtype option on import or set low_memory=False.
  crash_df = pd.read_csv('datasets/Motor_Vehicle_Collisions_-_Crashes_20260407.csv', delimiter=',')
/var/folders/tb/jb06w8hn5r72136sx1ztbxnr0000gn/T/ipykernel_35980/1986786415.py:2: DtypeWarning: Columns (0: VEHICLE_MODEL, 1: VEHICLE_OCCUPANTS) have mixed types. Specify dtype option on import or set low_memory=False.
  vehicle_df = pd.read_csv('datasets/Motor_Vehicle_Collisions_-_Vehicles_20260503.csv', delimiter=',')
/var/folders/tb/jb06w8hn5r72136sx1ztbxnr0000gn/T/ipykernel_35980/1986786415.py:3: DtypeWarning: Columns (0: PERSON_AGE) have mixed types. Specify dtype option on import or set low_memory=False.
  person_df = pd.read_csv('datasets/Motor_Vehicle_Collisions_-_Person_20260503.csv', delimiter=',')


In [3]:
#The size of the dataset in MB
print(f"Size of the crash dataset: {crash_df.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")
print(f"Size of the vehicle dataset: {vehicle_df.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")
print(f"Size of the person dataset: {person_df.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")

Size of the crash dataset: 875.21 MB
Size of the vehicle dataset: 1605.91 MB
Size of the person dataset: 1883.19 MB


### Clean crash data

In [51]:
crash_clean_df = crash_df.copy()

In [52]:
crash_clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2253192 entries, 0 to 2253191
Data columns (total 29 columns):
 #   Column                         Dtype  
---  ------                         -----  
 0   CRASH DATE                     str    
 1   CRASH TIME                     str    
 2   BOROUGH                        str    
 3   ZIP CODE                       object 
 4   LATITUDE                       float64
 5   LONGITUDE                      float64
 6   LOCATION                       str    
 7   ON STREET NAME                 str    
 8   CROSS STREET NAME              str    
 9   OFF STREET NAME                str    
 10  NUMBER OF PERSONS INJURED      float64
 11  NUMBER OF PERSONS KILLED       float64
 12  NUMBER OF PEDESTRIANS INJURED  int64  
 13  NUMBER OF PEDESTRIANS KILLED   int64  
 14  NUMBER OF CYCLIST INJURED      int64  
 15  NUMBER OF CYCLIST KILLED       int64  
 16  NUMBER OF MOTORIST INJURED     int64  
 17  NUMBER OF MOTORIST KILLED      int64  
 18  CONTRIBUTING 

In [53]:
# range of dates in the dataset
print(crash_clean_df["CRASH DATE"].min())
print(crash_clean_df["CRASH DATE"].max())

01/01/2013
12/31/2025


In [54]:
# how many nan values
crash_clean_df.isna().sum()

CRASH DATE                             0
CRASH TIME                             0
BOROUGH                           687462
ZIP CODE                          687745
LATITUDE                          240676
LONGITUDE                         240676
LOCATION                          240676
ON STREET NAME                    494034
CROSS STREET NAME                 862706
OFF STREET NAME                  1851532
NUMBER OF PERSONS INJURED             18
NUMBER OF PERSONS KILLED              31
NUMBER OF PEDESTRIANS INJURED          0
NUMBER OF PEDESTRIANS KILLED           0
NUMBER OF CYCLIST INJURED              0
NUMBER OF CYCLIST KILLED               0
NUMBER OF MOTORIST INJURED             0
NUMBER OF MOTORIST KILLED              0
CONTRIBUTING FACTOR VEHICLE 1       8152
CONTRIBUTING FACTOR VEHICLE 2     365477
CONTRIBUTING FACTOR VEHICLE 3    2090099
CONTRIBUTING FACTOR VEHICLE 4    2215899
CONTRIBUTING FACTOR VEHICLE 5    2242979
COLLISION_ID                           0
VEHICLE TYPE COD

In [55]:
print(crash_clean_df['ZIP CODE'].nunique())
print(crash_clean_df['BOROUGH'].nunique())

438
5


In [56]:
# are the missing values in zip code and in borough are the same rows? what about lat lon columns?
missing_zip = crash_clean_df['ZIP CODE'].isna()
missing_borough = crash_clean_df['BOROUGH'].isna()
missing_lat = crash_clean_df['LATITUDE'].isna()
missing_lon = crash_clean_df['LONGITUDE'].isna()
print((missing_zip != missing_borough).sum())
print((missing_zip != missing_lat).sum())
print((missing_zip != missing_lon).sum())
print((missing_borough != missing_lat).sum())

283
522727
522727
522496


#### Dropping nan values

As we will be working with geographical features of the data, we need to have consistency across columns such as borough, zip code, and coordinates - rows with NaNs in coordinates will be dropped, nans in zip code and borough could be inferred from the coordinates, as most of the rows with missing lat and lon have non-nan values.

For the features describing the collisions, we are not dropping NaN values,as it is expected that some refering to several vehicles will be empty, if it was a one-vehicle accident.

We will also drop the rows with nan values in number of persons injured and killed, as these features should be fully usable.

As the number of missing values in the columns containing street info would significantly further reduce the size of the dataset, while it is not sure if these features will be used, we decide to drop these features.

Nan values in contributing factor columns and vehicle type codes will be left unchanged, as it is expected for many values to be missing (not all crashes involve cars or a specific number of cars)

HERE COULD BE FILLING MISSING BOROUGHS AND ZIP CODES

In [57]:
crash_clean_df = crash_clean_df.dropna(subset=['ZIP CODE', 'LATITUDE', 'LONGITUDE', 'BOROUGH', 'NUMBER OF PERSONS INJURED', 'NUMBER OF PERSONS KILLED'])
crash_clean_df = crash_clean_df.drop(['ON STREET NAME', 'CROSS STREET NAME', 'OFF STREET NAME'], axis=1)
crash_clean_df.isna().sum()

CRASH DATE                             0
CRASH TIME                             0
BOROUGH                                0
ZIP CODE                               0
LATITUDE                               0
LONGITUDE                              0
LOCATION                               0
NUMBER OF PERSONS INJURED              0
NUMBER OF PERSONS KILLED               0
NUMBER OF PEDESTRIANS INJURED          0
NUMBER OF PEDESTRIANS KILLED           0
NUMBER OF CYCLIST INJURED              0
NUMBER OF CYCLIST KILLED               0
NUMBER OF MOTORIST INJURED             0
NUMBER OF MOTORIST KILLED              0
CONTRIBUTING FACTOR VEHICLE 1       6229
CONTRIBUTING FACTOR VEHICLE 2     263071
CONTRIBUTING FACTOR VEHICLE 3    1430720
CONTRIBUTING FACTOR VEHICLE 4    1504474
CONTRIBUTING FACTOR VEHICLE 5    1520950
COLLISION_ID                           0
VEHICLE TYPE CODE 1                12444
VEHICLE TYPE CODE 2               326583
VEHICLE TYPE CODE 3              1434426
VEHICLE TYPE COD

In [58]:
# remove duplicates
crash_clean_df = crash_clean_df.drop_duplicates()

In [59]:
print(crash_clean_df['ZIP CODE'].unique())

[11230.0 11208.0 11233.0 10475.0 11207.0 10017.0 11413.0 11434.0 11217.0
 11226.0 10463.0 10001.0 11372.0 10301.0 11215.0 11211.0 10455.0 11385.0
 11418.0 11225.0 11220.0 11411.0 10452.0 10466.0 10453.0 10019.0 11221.0
 11203.0 11419.0 11101.0 11106.0 11223.0 11422.0 11213.0 10128.0 11218.0
 11692.0 11420.0 11205.0 11212.0 10022.0 10011.0 10314.0 10461.0 11004.0
 10025.0 11373.0 10018.0 11234.0 10462.0 10472.0 11206.0 11236.0 11210.0
 11238.0 11209.0 10065.0 11249.0 11432.0 10032.0 11104.0 10002.0 10456.0
 10468.0 11201.0 11219.0 11235.0 10012.0 10305.0 10024.0 10458.0 11228.0
 11361.0 10035.0 11354.0 11377.0 11374.0 10467.0 11433.0 10016.0 10013.0
 11369.0 10457.0 10027.0 10028.0 11691.0 10014.0 10310.0 11231.0 10469.0
 10033.0 11435.0 10304.0 10459.0 10306.0 11428.0 10474.0 11356.0 11416.0
 11222.0 10168.0 10464.0 11375.0 10470.0 11366.0 10473.0 11421.0 11229.0
 11204.0 10031.0 11368.0 10029.0 10312.0 10460.0 10026.0 10021.0 10038.0
 11412.0 11105.0 11430.0 10039.0 11239.0 11367.0 11

In [60]:
# zip code cleanup, take only valid first 5 digits of a zip codes beginnig with 10 or 11, and convert to int
crash_clean_df = crash_clean_df[crash_clean_df['ZIP CODE'].str.match(r'^(10|11)\d{3}$', na=False)]
crash_clean_df['ZIP CODE'] = crash_clean_df['ZIP CODE'].astype(int)

In [61]:
crash_clean_df['ZIP CODE'].nunique()

203

In [62]:
# for convenience we will convert the date and time columns to datetime format
crash_clean_df['CRASH DATE'] = pd.to_datetime(crash_clean_df['CRASH DATE']).dt.date
crash_clean_df['CRASH TIME'] = pd.to_datetime(crash_clean_df['CRASH TIME']).dt.time
crash_clean_df

/var/folders/tb/jb06w8hn5r72136sx1ztbxnr0000gn/T/ipykernel_35980/2091243873.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  crash_clean_df['CRASH TIME'] = pd.to_datetime(crash_clean_df['CRASH TIME']).dt.time


,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,NUMBER OF PERSONS INJURED,NUMBER OF PERSONS KILLED,NUMBER OF PEDESTRIANS INJURED,NUMBER OF PEDESTRIANS KILLED,NUMBER OF CYCLIST INJURED,NUMBER OF CYCLIST KILLED,NUMBER OF MOTORIST INJURED,NUMBER OF MOTORIST KILLED,CONTRIBUTING FACTOR VEHICLE 1,CONTRIBUTING FACTOR VEHICLE 2,CONTRIBUTING FACTOR VEHICLE 3,CONTRIBUTING FACTOR VEHICLE 4,CONTRIBUTING FACTOR VEHICLE 5,COLLISION_ID,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5
1540097,2015-03-24,17:20:00,BROOKLYN,11214,40.597267,-73.998657,"(40.5972673, -73.9986569)",0.0,0.0,0,0,0,0,0,0,Unspecified,Unspecified,NaN,NaN,NaN,3191394,PASSENGER VEHICLE,PASSENGER VEHICLE,NaN,NaN,NaN
1540098,2015-04-11,15:50:00,BROOKLYN,11208,40.674384,-73.878960,"(40.6743843, -73.8789598)",0.0,0.0,0,0,0,0,0,0,Unspecified,Unspecified,NaN,NaN,NaN,3201359,SPORT UTILITY / STATION WAGON,SPORT UTILITY / STATION WAGON,NaN,NaN,NaN
1540102,2015-04-03,11:54:00,BRONX,10467,40.881784,-73.865347,"(40.8817836, -73.8653465)",0.0,0.0,0,0,0,0,0,0,Unspecified,Unspecified,NaN,NaN,NaN,3196959,UNKNOWN,UNKNOWN,NaN,NaN,NaN
1540104,2015-03-23,21:08:00,QUEENS,11355,40.758741,-73.814328,"(40.7587409, -73.8143276)",0.0,0.0,0,0,0,0,0,0,Backing Unsafely,Unspecified,NaN,NaN,NaN,3191031,PASSENGER VEHICLE,SPORT UTILITY / STATION WAGON,NaN,NaN,NaN
1540105,2015-04-17,08:43:00,BROOKLYN,11213,40.673860,-73.941775,"(40.6738596, -73.941775)",0.0,0.0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,3204178,PASSENGER VEHICLE,PASSENGER VEHICLE,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2129909,2024-10-25,14:54:00,BROOKLYN,11203,40.639190,-73.930200,"(40.63919, -73.9302)",0.0,0.0,0,0,0,0,0,0,Unspecified,Unspecified,NaN,NaN,NaN,4766637,Sedan,Bus,NaN,NaN,NaN
2129910,2024-10-24,12:02:00,MANHATTAN,10003,40.736960,-73.992920,"(40.73696, -73.99292)",0.0,0.0,0,0,0,0,0,0,Passing Too Closely,Unspecified,NaN,NaN,NaN,4766069,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,NaN
2129916,2024-10-24,21:30:00,QUEENS,11375,40.728287,-73.847220,"(40.728287, -73.84722)",0.0,0.0,0,0,0,0,0,0,Driver Inattention/Distraction,Unspecified,NaN,NaN,NaN,4766776,Station Wagon/Sport Utility Vehicle,Sedan,NaN,NaN,NaN
2129917,2024-10-24,17:40:00,BRONX,10472,40.828300,-73.882960,"(40.8283, -73.88296)",0.0,0.0,0,0,0,0,0,0,Driver Inattention/Distraction,Unspecified,NaN,NaN,NaN,4766166,Sedan,Motorcycle,NaN,NaN,NaN


### Clean vehicle data

In [63]:
vehicle_filtered_df = vehicle_df[vehicle_df['COLLISION_ID'].isin(crash_clean_df['COLLISION_ID'])]
vehicle_filtered_df.info()

<class 'pandas.DataFrame'>
Index: 834078 entries, 0 to 4525538
Data columns (total 25 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   UNIQUE_ID                    834078 non-null  int64  
 1   COLLISION_ID                 834078 non-null  int64  
 2   CRASH_DATE                   834078 non-null  str    
 3   CRASH_TIME                   834078 non-null  str    
 4   VEHICLE_ID                   834078 non-null  str    
 5   STATE_REGISTRATION           824312 non-null  str    
 6   VEHICLE_TYPE                 827097 non-null  str    
 7   VEHICLE_MAKE                 32163 non-null   str    
 8   VEHICLE_MODEL                4 non-null       str    
 9   VEHICLE_YEAR                 31433 non-null   float64
 10  TRAVEL_DIRECTION             38376 non-null   str    
 11  VEHICLE_OCCUPANTS            35788 non-null   object 
 12  DRIVER_SEX                   25459 non-null   str    
 13  DRIVER_LICENSE

In [64]:
vehicle_filtered_df.isna().sum()

UNIQUE_ID                           0
COLLISION_ID                        0
CRASH_DATE                          0
CRASH_TIME                          0
VEHICLE_ID                          0
STATE_REGISTRATION               9766
VEHICLE_TYPE                     6981
VEHICLE_MAKE                   801915
VEHICLE_MODEL                  834074
VEHICLE_YEAR                   802645
TRAVEL_DIRECTION               795702
VEHICLE_OCCUPANTS              798290
DRIVER_SEX                     808619
DRIVER_LICENSE_STATUS          810890
DRIVER_LICENSE_JURISDICTION    811103
PRE_CRASH                      516414
POINT_OF_IMPACT                795997
VEHICLE_DAMAGE                 796557
VEHICLE_DAMAGE_1               808040
VEHICLE_DAMAGE_2               812439
VEHICLE_DAMAGE_3               815556
PUBLIC_PROPERTY_DAMAGE         792639
PUBLIC_PROPERTY_DAMAGE_TYPE    833459
CONTRIBUTING_FACTOR_1           10598
CONTRIBUTING_FACTOR_2          796080
dtype: int64

Lot of missing data, but the dataset is big - more than 4 million rows, so the columns will not be dropped, as they might be needed differently based on the analysis.

In [65]:
# for convenience we will convert the date and time columns to datetime format
vehicle_filtered_df['CRASH_DATE'] = pd.to_datetime(vehicle_filtered_df['CRASH_DATE']).dt.date
vehicle_filtered_df['CRASH_TIME'] = pd.to_datetime(vehicle_filtered_df['CRASH_TIME']).dt.time
vehicle_filtered_df

/var/folders/tb/jb06w8hn5r72136sx1ztbxnr0000gn/T/ipykernel_35980/236786060.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  vehicle_filtered_df['CRASH_TIME'] = pd.to_datetime(vehicle_filtered_df['CRASH_TIME']).dt.time


,UNIQUE_ID,COLLISION_ID,CRASH_DATE,CRASH_TIME,VEHICLE_ID,STATE_REGISTRATION,VEHICLE_TYPE,VEHICLE_MAKE,VEHICLE_MODEL,VEHICLE_YEAR,TRAVEL_DIRECTION,VEHICLE_OCCUPANTS,DRIVER_SEX,DRIVER_LICENSE_STATUS,DRIVER_LICENSE_JURISDICTION,PRE_CRASH,POINT_OF_IMPACT,VEHICLE_DAMAGE,VEHICLE_DAMAGE_1,VEHICLE_DAMAGE_2,VEHICLE_DAMAGE_3,PUBLIC_PROPERTY_DAMAGE,PUBLIC_PROPERTY_DAMAGE_TYPE,CONTRIBUTING_FACTOR_1,CONTRIBUTING_FACTOR_2
0,10385780,100201,2012-09-07,09:03:00,1,NY,PASSENGER VEHICLE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,NaN
8,12254536,196425,2013-07-16,11:20:00,1,NY,PASSENGER VEHICLE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,NaN
15,11912713,176016,2012-08-11,19:36:00,2,NY,BICYCLE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,NaN
17,9879462,79561,2013-04-09,15:10:00,1,NY,PASSENGER VEHICLE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,NaN
21,8704041,19615,2013-07-16,17:10:00,1,NY,SPORT UTILITY / STATION WAGON,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4514173,21153874,4763359,2024-10-09,03:08:00,889ac359-7ec4-4a3d-aaf4-73213568bdf3,NY,Station Wagon/Sport Utility Vehicle,MITS -CAR/SUV,NaN,2022.0,East,2.0,M,Licensed,NY,Police Pursuit,Left Rear Bumper,Left Rear Bumper,Left Rear Quarter Panel,No Damage,No Damage,N,NaN,Unspecified,Unspecified
4515128,21153966,4784371,2025-01-07,06:00:00,e61defed-e386-49ce-8b76-8fdb5f15dd78,NY,Sedan,NISS -CAR/SUV,NaN,2010.0,North,0.0,F,Licensed,NY,Parked,Right Rear Bumper,Right Rear Bumper,No Damage,No Damage,No Damage,N,NaN,Unspecified,Unspecified
4515722,21153967,4784371,2025-01-07,06:00:00,da1356b5-8dcb-4c39-82f4-8e11d1ea6d68,NaN,NaN,NaN,NaN,NaN,North,NaN,NaN,NaN,NaN,Going Straight Ahead,NaN,NaN,NaN,NaN,NaN,Unspecified,NaN,Unspecified,Unspecified
4525360,21165972,4745021,2024-07-15,12:30:00,5a44af63-9884-4828-9250-d70820321997,NaN,Bike,NaN,NaN,NaN,East,1.0,M,Unlicensed,NY,Going Straight Ahead,Center Front End,No Damage,No Damage,No Damage,No Damage,N,NaN,Driver Inattention/Distraction,Unspecified


In [66]:
vehicle_filtered_df['VEHICLE_TYPE'].nunique()

288

In [67]:
# vehicle clean up

def clean_occupants(val):
    if pd.isna(val):
        return pd.NA
    
    # Convert to string, strip whitespace
    val = str(val).strip()
    
    # Empty string
    if val == "":
        return pd.NA
    
    # Remove commas from numbers like '999,999' or '1,355'
    val = val.replace(",", "")
    
    # Try converting to float first (handles '1.0' style strings)
    try:
        val = float(val)
    except ValueError:
        return pd.NA
    
    # Cap unrealistic values
    if val > 1000 or val < 0:
        return pd.NA
    
    return int(val)

vehicle_filtered_df["VEHICLE_OCCUPANTS"] = vehicle_filtered_df["VEHICLE_OCCUPANTS"].apply(clean_occupants).astype("Int64")

In [21]:
vehicle_filtered_df['VEHICLE_OCCUPANTS'].unique()

<IntegerArray>
[<NA>,    2,    1,    0,    3,    4,    5,    9,    8,   10,   11,    6,   12,
   15,    7,   17,   19,   16,   13,   14,   36,   29,   30,   22,   18,   32,
   20,   26]
Length: 28, dtype: Int64

In [68]:
# remove duplicates
vehicle_filtered_df = vehicle_filtered_df.drop_duplicates()

### Clean person data

In [69]:
person_filtered_df = person_df[person_df['COLLISION_ID'].isin(crash_clean_df['COLLISION_ID'])]
person_filtered_df.info()

<class 'pandas.DataFrame'>
Index: 170756 entries, 3113 to 5939971
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   UNIQUE_ID              170756 non-null  int64  
 1   COLLISION_ID           170756 non-null  int64  
 2   CRASH_DATE             170756 non-null  str    
 3   CRASH_TIME             170756 non-null  str    
 4   PERSON_ID              170756 non-null  str    
 5   PERSON_TYPE            170756 non-null  str    
 6   PERSON_INJURY          170756 non-null  str    
 7   VEHICLE_ID             139904 non-null  float64
 8   PERSON_AGE             159262 non-null  object 
 9   EJECTION               34862 non-null   str    
 10  EMOTIONAL_STATUS       37455 non-null   str    
 11  BODILY_INJURY          37455 non-null   str    
 12  POSITION_IN_VEHICLE    34866 non-null   str    
 13  SAFETY_EQUIPMENT       29873 non-null   str    
 14  PED_LOCATION           2690 non-null    str    


In [70]:
person_filtered_df.isna().sum()

UNIQUE_ID                     0
COLLISION_ID                  0
CRASH_DATE                    0
CRASH_TIME                    0
PERSON_ID                     0
PERSON_TYPE                   0
PERSON_INJURY                 0
VEHICLE_ID                30852
PERSON_AGE                11494
EJECTION                 135894
EMOTIONAL_STATUS         133301
BODILY_INJURY            133301
POSITION_IN_VEHICLE      135890
SAFETY_EQUIPMENT         140883
PED_LOCATION             168066
PED_ACTION               168066
COMPLAINT                133301
PED_ROLE                 102608
CONTRIBUTING_FACTOR_1    168080
CONTRIBUTING_FACTOR_2    168083
PERSON_SEX               109219
dtype: int64

Same as with the vehicle dataset, lot of missing values in many columns, but we will handle them accordingly to analysis needs.

In [71]:
# for convenience we will convert the date and time columns to datetime format
person_filtered_df['CRASH_DATE'] = pd.to_datetime(person_filtered_df['CRASH_DATE']).dt.date
person_filtered_df['CRASH_TIME'] = pd.to_datetime(person_filtered_df['CRASH_TIME']).dt.time
person_filtered_df

/var/folders/tb/jb06w8hn5r72136sx1ztbxnr0000gn/T/ipykernel_35980/1529943711.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  person_filtered_df['CRASH_TIME'] = pd.to_datetime(person_filtered_df['CRASH_TIME']).dt.time


,UNIQUE_ID,COLLISION_ID,CRASH_DATE,CRASH_TIME,PERSON_ID,PERSON_TYPE,PERSON_INJURY,VEHICLE_ID,PERSON_AGE,EJECTION,EMOTIONAL_STATUS,BODILY_INJURY,POSITION_IN_VEHICLE,SAFETY_EQUIPMENT,PED_LOCATION,PED_ACTION,COMPLAINT,PED_ROLE,CONTRIBUTING_FACTOR_1,CONTRIBUTING_FACTOR_2,PERSON_SEX
3113,2243130,3175479,2015-02-25,14:22:00,1,Pedestrian,Injured,NaN,69,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4167,1543324,100437,2012-10-27,03:44:00,2,Occupant,Injured,10386393.0,42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4237,1351006,49924,2013-09-06,06:00:00,1,Occupant,Injured,9283677.0,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5079,1989018,225562,2013-10-01,08:30:00,1,Occupant,Injured,12846899.0,42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5084,1874470,192348,2013-01-24,17:40:00,1,Pedestrian,Injured,NaN,57,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5922541,13851561,4784371,2025-01-07,06:00:00,6a246f54-0779-4d08-bf99-86f00c282ba0,Occupant,Unspecified,21153966.0,39,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Registrant,NaN,NaN,F
5939428,13872344,4745021,2024-07-15,12:30:00,d1b29055-7efa-43ba-8b55-976fa5e3dd03,Occupant,Unspecified,21165971.0,60.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Registrant,NaN,NaN,M
5939517,13872343,4745021,2024-07-15,12:30:00,55e3e536-3970-4d8a-9683-fb3497f526c5,Bicyclist,Injured,21165972.0,26.0,Not Ejected,Conscious,Knee-Lower Leg Foot,"Left rear passenger, or rear passenger on a bi...",NaN,NaN,NaN,Complaint of Pain or Nausea,Driver,NaN,NaN,M
5939603,13872341,4745021,2024-07-15,12:30:00,fbce2469-de46-40b4-ac3a-fc092a79ff2a,Occupant,Unspecified,21165971.0,60.0,Not Ejected,Does Not Apply,Does Not Apply,Driver,NaN,NaN,NaN,Does Not Apply,Driver,NaN,NaN,M


In [72]:
# remove duplicates
person_filtered_df = person_filtered_df.drop_duplicates()

In [74]:
person_filtered_df['PERSON_AGE'].unique()

array(['69', '42', '38', '57', '29', '23', '74', '17', '30', '7', '26',
       '53', '3', '37', '12', '35', '28', '20', '68', '88', '51', nan,
       '61', '56', '64', '19', '0', '70', '41', '47', '36', '49', '44',
       '32', '6', '52', '40', '75', '25', '33', '4', '10', '46', '22',
       '1', '59', '27', '66', '34', '48', '55', '72', '87', '39', '83',
       '18', '24', '11', '45', '54', '67', '2', '50', '16', '31', '60',
       '13', '62', '15', '21', '43', '65', '73', '58', '63', '86', '5',
       '14', '78', '77', '9', '89', '8', '71', '85', '76', '84', '100',
       '94', '79', '81', '90', '80', '149', '96', '82', '91', '97', '92',
       '148', '95', '98', '123', '928', '99', '999', '93', '120', '130',
       '140', '161', '332', '111', '128', '105', '110', '104', 7.0, 47.0,
       71.0, 49.0, 17.0, 39.0, 18.0, 38.0, 19.0, 73.0, 25.0, 57.0, 34.0,
       59.0, 41.0, 33.0, 26.0, 37.0, 22.0, 66.0, 48.0, 43.0, 27.0, 46.0,
       52.0, 35.0, 30.0, 76.0, 42.0, 36.0, 21.0, 14.0, 10.0

In [ ]:
# person age clean up

def clean_person_age(val):
    if pd.isna(val):
        return pd.NA
    
    # Convert to string, strip whitespace
    val = str(val).strip()
    
    # Empty string
    if val == "":
        return pd.NA
    
    # Remove commas from numbers like '999,999' or '1,355'
    val = val.replace(",", "")
    
    # Try converting to float first (handles '1.0' style strings)
    try:
        val = float(val)
    except ValueError:
        return pd.NA
    
    # Cap unrealistic values
    if val > 120 or val < 0:
        return pd.NA
    
    return int(val)

person_filtered_df["PERSON_AGE"] = person_filtered_df["PERSON_AGE"].apply(clean_person_age).astype("Int64")

### Merging


In [78]:
# compare vehicle_id in vehicle and person datasets
print(vehicle_filtered_df['VEHICLE_ID'].dtype)
print(person_filtered_df['VEHICLE_ID'].dtype)
vehicle_filtered_df.head()

str
float64


,UNIQUE_ID,COLLISION_ID,CRASH_DATE,CRASH_TIME,VEHICLE_ID,STATE_REGISTRATION,VEHICLE_TYPE,VEHICLE_MAKE,VEHICLE_MODEL,VEHICLE_YEAR,TRAVEL_DIRECTION,VEHICLE_OCCUPANTS,DRIVER_SEX,DRIVER_LICENSE_STATUS,DRIVER_LICENSE_JURISDICTION,PRE_CRASH,POINT_OF_IMPACT,VEHICLE_DAMAGE,VEHICLE_DAMAGE_1,VEHICLE_DAMAGE_2,VEHICLE_DAMAGE_3,PUBLIC_PROPERTY_DAMAGE,PUBLIC_PROPERTY_DAMAGE_TYPE,CONTRIBUTING_FACTOR_1,CONTRIBUTING_FACTOR_2
0,10385780,100201,2012-09-07,09:03:00,1,NY,PASSENGER VEHICLE,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,NaN
8,12254536,196425,2013-07-16,11:20:00,1,NY,PASSENGER VEHICLE,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,NaN
15,11912713,176016,2012-08-11,19:36:00,2,NY,BICYCLE,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,NaN
17,9879462,79561,2013-04-09,15:10:00,1,NY,PASSENGER VEHICLE,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,NaN
21,8704041,19615,2013-07-16,17:10:00,1,NY,SPORT UTILITY / STATION WAGON,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,NaN


In [79]:
person_filtered_df.head()

,UNIQUE_ID,COLLISION_ID,CRASH_DATE,CRASH_TIME,PERSON_ID,PERSON_TYPE,PERSON_INJURY,VEHICLE_ID,PERSON_AGE,EJECTION,EMOTIONAL_STATUS,BODILY_INJURY,POSITION_IN_VEHICLE,SAFETY_EQUIPMENT,PED_LOCATION,PED_ACTION,COMPLAINT,PED_ROLE,CONTRIBUTING_FACTOR_1,CONTRIBUTING_FACTOR_2,PERSON_SEX
3113,2243130,3175479,2015-02-25,14:22:00,1,Pedestrian,Injured,NaN,69,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4167,1543324,100437,2012-10-27,03:44:00,2,Occupant,Injured,10386393.0,42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4237,1351006,49924,2013-09-06,06:00:00,1,Occupant,Injured,9283677.0,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5079,1989018,225562,2013-10-01,08:30:00,1,Occupant,Injured,12846899.0,42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5084,1874470,192348,2013-01-24,17:40:00,1,Pedestrian,Injured,NaN,57,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [80]:
vehicle_filtered_df[vehicle_filtered_df['COLLISION_ID'] == 4229067]

,UNIQUE_ID,COLLISION_ID,CRASH_DATE,CRASH_TIME,VEHICLE_ID,STATE_REGISTRATION,VEHICLE_TYPE,VEHICLE_MAKE,VEHICLE_MODEL,VEHICLE_YEAR,TRAVEL_DIRECTION,VEHICLE_OCCUPANTS,DRIVER_SEX,DRIVER_LICENSE_STATUS,DRIVER_LICENSE_JURISDICTION,PRE_CRASH,POINT_OF_IMPACT,VEHICLE_DAMAGE,VEHICLE_DAMAGE_1,VEHICLE_DAMAGE_2,VEHICLE_DAMAGE_3,PUBLIC_PROPERTY_DAMAGE,PUBLIC_PROPERTY_DAMAGE_TYPE,CONTRIBUTING_FACTOR_1,CONTRIBUTING_FACTOR_2


In [81]:
person_filtered_df[person_filtered_df['COLLISION_ID'] == 4229067].head(5)

,UNIQUE_ID,COLLISION_ID,CRASH_DATE,CRASH_TIME,PERSON_ID,PERSON_TYPE,PERSON_INJURY,VEHICLE_ID,PERSON_AGE,EJECTION,EMOTIONAL_STATUS,BODILY_INJURY,POSITION_IN_VEHICLE,SAFETY_EQUIPMENT,PED_LOCATION,PED_ACTION,COMPLAINT,PED_ROLE,CONTRIBUTING_FACTOR_1,CONTRIBUTING_FACTOR_2,PERSON_SEX


Vehicle_id in person dataset is the unique_id in vehicle_id, so the datasets can be merged using this link.

In [82]:
print(person_filtered_df['VEHICLE_ID'].dtype)
print(vehicle_filtered_df['UNIQUE_ID'].dtype)

float64
int64


In [83]:
# convert vehicle_id in person dataset to int if the value is numeric, otherwise it will be converted to NaN
person_filtered_df['VEHICLE_ID'] = person_filtered_df['VEHICLE_ID'].astype('Int64')
print(person_filtered_df['VEHICLE_ID'].dtype)
print(vehicle_filtered_df['UNIQUE_ID'].dtype)

Int64
int64


In [84]:
# add to every column name in vehicle dataset the prefix 'v_' to avoid confusion when merging with person dataset
crash_clean_df = crash_clean_df.add_prefix('c_')
vehicle_filtered_df = vehicle_filtered_df.add_prefix('v_')
person_filtered_df = person_filtered_df.add_prefix('p_')

In [85]:
crash_vehicle_df = crash_clean_df.merge(vehicle_filtered_df, left_on="c_COLLISION_ID", right_on='v_COLLISION_ID',  how="left")

In [86]:
full_df = crash_vehicle_df.merge(
    person_filtered_df,
    left_on=["c_COLLISION_ID", "v_UNIQUE_ID"],
    right_on=["p_COLLISION_ID", "p_VEHICLE_ID"],
    how="left"
)
full_df.columns

Index(['c_CRASH DATE', 'c_CRASH TIME', 'c_BOROUGH', 'c_ZIP CODE', 'c_LATITUDE',
       'c_LONGITUDE', 'c_LOCATION', 'c_NUMBER OF PERSONS INJURED',
       'c_NUMBER OF PERSONS KILLED', 'c_NUMBER OF PEDESTRIANS INJURED',
       'c_NUMBER OF PEDESTRIANS KILLED', 'c_NUMBER OF CYCLIST INJURED',
       'c_NUMBER OF CYCLIST KILLED', 'c_NUMBER OF MOTORIST INJURED',
       'c_NUMBER OF MOTORIST KILLED', 'c_CONTRIBUTING FACTOR VEHICLE 1',
       'c_CONTRIBUTING FACTOR VEHICLE 2', 'c_CONTRIBUTING FACTOR VEHICLE 3',
       'c_CONTRIBUTING FACTOR VEHICLE 4', 'c_CONTRIBUTING FACTOR VEHICLE 5',
       'c_COLLISION_ID', 'c_VEHICLE TYPE CODE 1', 'c_VEHICLE TYPE CODE 2',
       'c_VEHICLE TYPE CODE 3', 'c_VEHICLE TYPE CODE 4',
       'c_VEHICLE TYPE CODE 5', 'v_UNIQUE_ID', 'v_COLLISION_ID',
       'v_CRASH_DATE', 'v_CRASH_TIME', 'v_VEHICLE_ID', 'v_STATE_REGISTRATION',
       'v_VEHICLE_TYPE', 'v_VEHICLE_MAKE', 'v_VEHICLE_MODEL', 'v_VEHICLE_YEAR',
       'v_TRAVEL_DIRECTION', 'v_VEHICLE_OCCUPANTS', 'v_DR

In [87]:
# Should match original crash count if all crashes are preserved
assert full_df["c_COLLISION_ID"].nunique() == crash_clean_df["c_COLLISION_ID"].nunique()

# Spot-check pedestrians: null p_VEHICLE_ID but valid collision_id
pedestrians = full_df[full_df["p_VEHICLE_ID"].isna()]
print(f"Pedestrian-involved rows: {len(pedestrians)}")

# Check for unexpected row explosion (a sign of a bad join)
print(f"Crashes: {len(crash_clean_df)}, Final rows: {len(full_df)}")

Pedestrian-involved rows: 741682
Crashes: 422534, Final rows: 881586


### Renaming columns and dropping duplicating data

In [88]:
full_df[['c_CRASH DATE', 'c_CRASH TIME', 'v_CRASH_DATE', 'v_CRASH_TIME', 'p_CRASH_DATE', 'p_CRASH_TIME']].isna().sum()

c_CRASH DATE         0
c_CRASH TIME         0
v_CRASH_DATE        18
v_CRASH_TIME        18
p_CRASH_DATE    741676
p_CRASH_TIME    741676
dtype: int64

In [89]:
full_df[['v_UNIQUE_ID', 'p_UNIQUE_ID']].isna().sum()

v_UNIQUE_ID        18
p_UNIQUE_ID    741676
dtype: int64

The missing values for vehicles, are the crashes that have no matching vehicle record. The missing values for person are the crash, vehicle pair that had no match to any (crash, vehicle) pair.

In [90]:
# flag columns to indicate if a crash has a vehicle or person record
full_df["HAS_VEHICLE"] = full_df["v_UNIQUE_ID"].notna().astype(int)
full_df["HAS_PERSON"] = full_df["p_UNIQUE_ID"].notna().astype(int)

In [91]:
# let's drop all spare columns
cols_to_drop = [
    "v_COLLISION_ID", "p_COLLISION_ID",
    "p_VEHICLE_ID", "v_VEHICLE_ID",     
    "v_CRASH_DATE", "v_CRASH_TIME",
    "p_CRASH_DATE", "p_CRASH_TIME",
]
full_df = full_df.drop(columns=cols_to_drop)

In [92]:
full_df.head()

,c_CRASH DATE,c_CRASH TIME,c_BOROUGH,c_ZIP CODE,c_LATITUDE,c_LONGITUDE,c_LOCATION,c_NUMBER OF PERSONS INJURED,c_NUMBER OF PERSONS KILLED,c_NUMBER OF PEDESTRIANS INJURED,c_NUMBER OF PEDESTRIANS KILLED,c_NUMBER OF CYCLIST INJURED,c_NUMBER OF CYCLIST KILLED,c_NUMBER OF MOTORIST INJURED,c_NUMBER OF MOTORIST KILLED,c_CONTRIBUTING FACTOR VEHICLE 1,c_CONTRIBUTING FACTOR VEHICLE 2,c_CONTRIBUTING FACTOR VEHICLE 3,c_CONTRIBUTING FACTOR VEHICLE 4,c_CONTRIBUTING FACTOR VEHICLE 5,c_COLLISION_ID,c_VEHICLE TYPE CODE 1,c_VEHICLE TYPE CODE 2,c_VEHICLE TYPE CODE 3,c_VEHICLE TYPE CODE 4,c_VEHICLE TYPE CODE 5,v_UNIQUE_ID,v_STATE_REGISTRATION,v_VEHICLE_TYPE,v_VEHICLE_MAKE,v_VEHICLE_MODEL,v_VEHICLE_YEAR,v_TRAVEL_DIRECTION,v_VEHICLE_OCCUPANTS,v_DRIVER_SEX,v_DRIVER_LICENSE_STATUS,v_DRIVER_LICENSE_JURISDICTION,v_PRE_CRASH,v_POINT_OF_IMPACT,v_VEHICLE_DAMAGE,v_VEHICLE_DAMAGE_1,v_VEHICLE_DAMAGE_2,v_VEHICLE_DAMAGE_3,v_PUBLIC_PROPERTY_DAMAGE,v_PUBLIC_PROPERTY_DAMAGE_TYPE,v_CONTRIBUTING_FACTOR_1,v_CONTRIBUTING_FACTOR_2,p_UNIQUE_ID,p_PERSON_ID,p_PERSON_TYPE,p_PERSON_INJURY,p_PERSON_AGE,p_EJECTION,p_EMOTIONAL_STATUS,p_BODILY_INJURY,p_POSITION_IN_VEHICLE,p_SAFETY_EQUIPMENT,p_PED_LOCATION,p_PED_ACTION,p_COMPLAINT,p_PED_ROLE,p_CONTRIBUTING_FACTOR_1,p_CONTRIBUTING_FACTOR_2,p_PERSON_SEX,HAS_VEHICLE,HAS_PERSON
0,2015-03-24,17:20:00,BROOKLYN,11214,40.597267,-73.998657,"(40.5972673, -73.9986569)",0.0,0.0,0,0,0,0,0,0,Unspecified,Unspecified,NaN,NaN,NaN,3191394,PASSENGER VEHICLE,PASSENGER VEHICLE,NaN,NaN,NaN,14627513.0,NY,PASSENGER VEHICLE,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,Stopped in Traffic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
1,2015-03-24,17:20:00,BROOKLYN,11214,40.597267,-73.998657,"(40.5972673, -73.9986569)",0.0,0.0,0,0,0,0,0,0,Unspecified,Unspecified,NaN,NaN,NaN,3191394,PASSENGER VEHICLE,PASSENGER VEHICLE,NaN,NaN,NaN,14627514.0,NY,PASSENGER VEHICLE,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,Going Straight Ahead,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
2,2015-04-11,15:50:00,BROOKLYN,11208,40.674384,-73.878960,"(40.6743843, -73.8789598)",0.0,0.0,0,0,0,0,0,0,Unspecified,Unspecified,NaN,NaN,NaN,3201359,SPORT UTILITY / STATION WAGON,SPORT UTILITY / STATION WAGON,NaN,NaN,NaN,14651243.0,NY,SPORT UTILITY / STATION WAGON,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,Backing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
3,2015-04-11,15:50:00,BROOKLYN,11208,40.674384,-73.878960,"(40.6743843, -73.8789598)",0.0,0.0,0,0,0,0,0,0,Unspecified,Unspecified,NaN,NaN,NaN,3201359,SPORT UTILITY / STATION WAGON,SPORT UTILITY / STATION WAGON,NaN,NaN,NaN,14651242.0,NY,SPORT UTILITY / STATION WAGON,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,Going Straight Ahead,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
4,2015-04-03,11:54:00,BRONX,10467,40.881784,-73.865347,"(40.8817836, -73.8653465)",0.0,0.0,0,0,0,0,0,0,Unspecified,Unspecified,NaN,NaN,NaN,3196959,UNKNOWN,UNKNOWN,NaN,NaN,NaN,14641001.0,NY,UNKNOWN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,Going Straight Ahead,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0


In [93]:
# save as a new csv file
full_df.to_csv('datasets/crash_vehicle_person_merged_data.csv', index=False)

In [94]:
# save to parquet
full_df.to_parquet('datasets/crash_vehicle_person_merged_data.parquet', engine= 'pyarrow', index=False)

In [96]:
# save the list of columns to a text file
with open('datasets/merged_data_columns_list.txt', 'w') as f:
    for col in full_df.columns:
        f.write(f"{col}\n")

In [95]:
full_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 881586 entries, 0 to 881585
Data columns (total 66 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   c_CRASH DATE                     881586 non-null  object 
 1   c_CRASH TIME                     881586 non-null  object 
 2   c_BOROUGH                        881586 non-null  str    
 3   c_ZIP CODE                       881586 non-null  int64  
 4   c_LATITUDE                       881586 non-null  float64
 5   c_LONGITUDE                      881586 non-null  float64
 6   c_LOCATION                       881586 non-null  str    
 7   c_NUMBER OF PERSONS INJURED      881586 non-null  float64
 8   c_NUMBER OF PERSONS KILLED       881586 non-null  float64
 9   c_NUMBER OF PEDESTRIANS INJURED  881586 non-null  int64  
 10  c_NUMBER OF PEDESTRIANS KILLED   881586 non-null  int64  
 11  c_NUMBER OF CYCLIST INJURED      881586 non-null  int64  
 12  c_NUMBER OF C